Modeling Features: Mass Stellar Dark
====================================

The majority of example scripts fit a mass profile which represents the _total_ mass of the lens galaxy (its stars,
dark matter and other components combined). This typically uses an `Isothermal` or `PowerLaw` mass profile.

This script fits a mass model which decomposes the lens galaxy's mass into its stars and dark matter.

__Practical Use: Read This First__

This script is a tutorial. It produces a working fit by "cheating" — every prior is initialised at the true
simulator value, narrowed by a small Gaussian. On real data this is impossible, and a single Nautilus search on
a decomposed stellar + dark matter mass model would almost certainly converge to a local maximum. The mass-to-light
coupling between the lens light and its stellar mass component makes the parameter degeneracies more severe than
in total-mass fits: when the M/L ratio drifts, the stellar mass deflection drifts with it, and the dark NFW
component has to absorb the slack — these directions are hard for a non-linear search to disentangle without a
good starting point.

The script you will actually use to fit a decomposed-mass model on real data is
`autolens_workspace/scripts/imaging/features/advanced/mass_stellar_dark/chaining.py`, which runs two chained
non-linear searches: the first fits the lens light alone (treating it as a pure light profile), the second
introduces the stellar-mass coupling by re-using the bulge result as a `lmp.Sersic` and adds the dark NFW. This
gives the second search a tight prior on the bulge geometry, which is the parameter set most strongly tied to
the stellar mass deflection.

For production-quality modeling, see `slam.py` in the same directory, which uses the `MASS_LIGHT_DARK` SLaM
pipeline to chain through SOURCE LP → SOURCE PIX → LIGHT LP → MASS_LIGHT_DARK and ends with a pixelized source
reconstruction.

Read this script to understand the model composition API, then jump to `chaining.py`.

__Contents__

- **Advantages & Disadvantages:** Decomposed mass models measure direct properties of the stars and dark matter, for example the.
- **Model:** Compose the lens model fitted to the data.
- **Dataset & Mask:** Standard set up of the dataset and mask that is fitted.
- **Over Sampling:** Set up the adaptive over-sampling grid for accurate light profile evaluation.
- **Model Cookbook:** A full description of model composition is provided by the model cookbook.
- **Search:** Configure the non-linear search used to fit the model.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **VRAM:** The `modeling` example explains how VRAM is used during GPU-based fitting and how to print the.
- **Run Time:** Profiling the expected run time of the model-fit.
- **Result:** Overview of the results of the model-fit.
- **Wrap Up:** Summary of the script and next steps.

__Advantages__

Decomposed mass models measure direct properties of the stars and dark matter, for example the lens's stellar mass,
dark matter mass and the relative distribution between the two. Total mass profiles only inform us about the
superposition of these two components.

Decomposed mass models couple the lens galaxy's light profile to its stellar mass distribution, meaning that
additional information in the lens galaxy emission is used to constrain the mass model. Whilst total mass models
also fit the lens light, they do not couple it to the mass model and thus do not exploit this extra information.

Total mass models like the `Isothermal` and `PowerLaw` assume that the overall mass distribution of the lens galaxy
can be described using a single elliptical coordinate system. The stellar and dark components of a decomposed mass
model each have their own elliptical coordinate system, meaning that the mass model can be more complex and accurate.

__Disadvantages__

Assumptions must be made about how light and mass are coupled. This script assumes a constant mass-to-light raito,
however it is not clear this is a reliable assumption in many lens galaxies.

**PyAutoLens** supports more complex mass models which introduce a radial gradient into the mass-to-light ratio.
However, these are more complex and therefore are difficult to fit robustly. Furthermore, it is still not clear
whether the way they couple light to mass is a reliable assumption.

Performing ray-tracing with decomposed mass models is also more computationally expensive, meaning that the run times
of model-fits using these models is typically longer than total mass models.

__Model__

This script fits an `Imaging` dataset of a 'galaxy-scale' strong lens with a model where:

 - The lens galaxy's light is a linear `Sersic`.
 - The lens galaxy's stellar mass distribution is tied to the light model above.
 - The lens galaxy's dark matter mass distribution is a `NFW`.
 - The source galaxy's light is a Multi Gaussian Expansion.

__Start Here Notebook__

If any code in this script is unclear, refer to the `imaging/start_here.ipynb` notebook.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Dataset__

Load and plot the strong lens dataset `mass_stellar_dark` via .fits files.

In [ ]:
dataset_name = "mass_stellar_dark"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable,
            "scripts/imaging/features/advanced/mass_stellar_dark/simulator.py",
        ],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

__Mask__

Define a 3.0" circular mask, which includes the emission of the lens and source galaxies.

In [ ]:
mask_radius = 3.0

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=mask_radius,
)

dataset = dataset.apply_mask(mask=mask)

aplt.subplot_imaging_dataset(dataset=dataset)

__Over Sampling__

Apply adaptive over sampling to ensure the lens galaxy light calculation is accurate, you can read up on over-sampling 
in more detail via the `autolens_workspace/*/guides/advanced/over_sampling.ipynb` notebook.

In [ ]:
over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=dataset.grid,
    sub_size_list=[4, 2, 2],
    radial_list=[0.3, 0.6],
    centre_list=[(0.0, 0.0)],
)

dataset = dataset.apply_over_sampling(over_sample_size_lp=over_sample_size)

aplt.subplot_imaging_dataset(dataset=dataset)

__Model__

We compose a lens model where:

 - The lens galaxy's light and stellar mass is a linear `Sersic` [7 parameters].

 - The lens galaxy's dark matter mass distribution is a `NFW` whose centre is aligned with the 
 `Sersic` bulge of the light and stellar mass model above [5 parameters].

 - The system's external shear is an `ExternalShear` held in a `MassField` [2 parameters].

 - The source galaxy's light is a Multi Gaussian Expansion [4 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=22.

Note that for the stellar light and mass, we are using a "light and mass profile" via the `.lmp` package. This
profiles simultaneously acts like a light and mass profile.

For the dark matter, we use an `NFW`, which is a common mass profile to represent dark matter.

__Model Cookbook__

A full description of model composition is provided by the model cookbook: 

https://pyautolens.readthedocs.io/en/latest/general/model_cookbook.html

In [ ]:
# Lens:

bulge = af.Model(al.lmp.Sersic)
dark = af.Model(al.mp.NFW)
bulge.centre = dark.centre
lens = af.Model(al.Galaxy, redshift=0.5, bulge=bulge, dark=dark)

# External Shear:

field = af.Model(al.MassField, redshift=0.5, shear=af.Model(al.mp.ExternalShear))

# Source:

bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,
)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

# Overall Model:

model = af.Collection(
    galaxies=af.Collection(lens=lens, source=source),
    fields=field,
)

The `info` attribute shows the model in a readable format (if this does not display clearly on your screen refer to
`start_here.ipynb` for a description of how to fix this).

This confirms that the lens model has both a `Sersic` light and mass profile and `NFW` dark matter profile, which 
are aligned.

In [ ]:
print(model.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free, 
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the 
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed 
parameter.

In [ ]:
af.ModelPlotter(model).figure()

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start.here.py` for a full 
description).

In [ ]:
search = af.Nautilus(
    path_prefix=Path("imaging") / "features",
    name="mass_stellar_dark",
    unique_tag=dataset_name,
    n_live=150,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Analysis__

Create the `AnalysisImaging` object defining how the via Nautilus the model is fitted to the data.

In [ ]:
analysis = al.AnalysisImaging(dataset=dataset, use_jax=True)

__VRAM__

The `modeling` example explains how VRAM is used during GPU-based fitting and how to print the estimated VRAM
required by a model.

Deflection angle calculations of stellar mass models and dark matter mass models can use techniques whichs
store more data in VRAM than other methods. 

Given VRAM use is an important consideration, we print out the estimated VRAM required for this
model-fit and advise you do this for your own double source plane lens model-fits.

The method below prints the VRAM usage estimate for the analysis and model with the specified batch size,
it takes about 20-30 seconds to run so you may want to comment it out once you are familiar with your GPU's VRAM limits.

In [ ]:
analysis.print_vram_use(model=model, batch_size=search.batch_size)

__Run Time__

The likelihood evaluation time for analysing stellar and dark matter mass models is longer than for total mass models
like the isothermal or power-law. This is because the deflection angles of these mass profiles are more expensive to
compute, requiring a Gaussian expansion or numerical calculation.

However, they have far fewer parameters than total mass models, when those models are also modeling the lens light. 
This is because many of the light and mass profile parameters are shared and fitted for simultaneously, reducing the
overall dimensionality of non-linear parameter space.

__Model-Fit__

We begin the model-fit by passing the model and analysis object to the non-linear search (checkout the output folder
for on-the-fly visualization and results).

In [ ]:
result = search.fit(model=model, analysis=analysis)

__Result__

The `info` attribute shows the model in a readable format (if this does not display clearly on your screen refer to
`start_here.ipynb` for a description of how to fix this).

In [ ]:
print(result.info)

We plot the maximum likelihood fit, tracer images and posteriors inferred via Nautilus.

These plots show that a decomposed stars and dark matter model is still able to produce ray-tracing and
the lensed source's emission.

In [ ]:
print(result.max_log_likelihood_instance)

aplt.subplot_tracer(tracer=result.max_log_likelihood_tracer, grid=result.grids.lp)

aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)

aplt.corner_anesthetic(samples=result.samples)

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.

These examples include a results API with specific tools for visualizing and analysing decomposed mass model,
for example 1D plots which separately show the density of stars and dark matter as a function of radius.

__Wrap Up__

Decomposed mass models have advantages and disavantages compared to total mass models.

The model which is best suited to your needs depends on the science you are hoping to undertake and the quality of the
data you are fitting.

In general, it is recommended that you first get fits going using total mass models, because they are simpler and make
fewer assumptions regarding how light is tied to mass. Once you have robust results, decomposed mass models can then
be fitted and compared in order to gain deeper insight.